# Date — Canonicalizing date values with Paxman

**Domain:** date  
**Capability contract:** `Date` (frozen `CanonicalDateContract`)  
**Public API:** `paxman.canonicalize(input_data, contract)`

Follows the **reference template** (`00_email.ipynb`). Run every cell in order; no hidden state. Date canonicalizes many date/datetime spellings into `YYYY-MM-DD` (or `...T...Z`). When a format is ambiguous, Paxman surfaces the readings instead of guessing.

> Run every cell in order. `artifact.value` holds the canonical string when `artifact.status == "CANONICALIZED"`; otherwise it is `None`.

In [ ]:
from paxman import canonicalize, Date, ContractError, CanonicalizationError

def show(raw, contract):
    """Canonicalize `raw`; print status + value."""
    artifact = canonicalize(raw, contract)
    if artifact.status.name == "CANONICALIZED":
        print(f"{raw!r:38} -> {artifact.status.name:14} {artifact.value!r}")
    else:
        print(f"{raw!r:38} -> {artifact.status.name:14} (no canonical value)")
    return artifact

With an explicit `locale`, slash forms resolve deterministically: `US` = MM/DD, `EU` = DD/MM.

In [ ]:
show("2025-03-04", Date(locale="US"))
show("03/04/2025", Date(locale="US"))   # MM/DD
show("03/04/2025", Date(locale="EU"))   # DD/MM
show("16 July 2026", Date(language="en"))

Datetimes normalize to RFC 3339 UTC (`Z`). A *naive* datetime (no zone) is `AMBIGUOUS` per RFC 3339 §5.6 — Paxman will not assume a zone.

In [ ]:
show("2025-01-01T07:00:00-05:00", Date(locale="ISO"))
show("2025-01-01T07:00:00", Date(locale="ISO"))   # AMBIGUOUS

Ambiguity is surfaced, never guessed: `locale="ISO"` leaves MM/DD vs DD/MM open; a 2-digit year with no policy spans centuries.

In [ ]:
show("03/04/2025", Date(locale="ISO"))   # AMBIGUOUS
show("03/04/26", Date(locale="US"))      # AMBIGUOUS
show("not a date", Date(locale="US"))    # INVALID

## Where to go next

- **`10_engine.ipynb`** — `Engine.default()`, `Engine.with_authorities(...)`, `canonicalize_with(...)`, and the `authority_override` escape hatch.
- **`11_dsl.ipynb`** — build contracts from a DSL string with `parse_contract`.
- All inputs above are sourced from `NOTEBOOK_INPUTS.md` (verified against the working tree).

> Every other capability notebook (Boolean, Country, Geolocation, IP, Money, Phone, URL, UUID) follows this exact structure.